# 벡터 DB 생성 및 적재를 위한 노트북 파일입니다.

### step1 - 필요한 데이터 경로 가져오기

In [4]:
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Load the CSV from the notebook parent directory.
csv_path = Path("../museum_final_738.csv")
df = pd.read_csv(csv_path, encoding="utf-8-sig")

#테스트
display(df.head(2))


,museum_id,name,ownership_type,sido,sigungu,address,latitude,longitude,operator_name,manager_name,...,official_verified_field_count,text,text_length,text_sha256,source_doc_count,source_types,source_documents_json,text_spans_json,removed_duplicate_sentence_count,merge_review_flags
0,M0001,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,경상남도 창녕군청,경상남도 창녕군청,...,0.0,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,407,1fcf9c09a671bb09df9f0a8d84edf4d13a2fd68b08c0c4...,1,TOUR_API,"[{""doc_id"":""DOC-B0001-TOUR_API-2785601"",""sourc...","[{""start"":0,""end"":94,""doc_ids"":[""DOC-B0001-TOU...",0,NaN
1,M0003,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,경상남도 창녕군청,경상남도 창녕군청,...,0.0,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,608,35480738914dd66b95a1bef1f432ae7ce951e01ee9503d...,1,TOUR_API,"[{""doc_id"":""DOC-B0003-TOUR_API-130107"",""source...","[{""start"":0,""end"":60,""doc_ids"":[""DOC-B0003-TOU...",0,NaN


### step2 - 임베딩 모델 가져오기(text-embedding-3-small)

In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

env_path = next((parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents] if (parent / ".env").is_file()), None)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in .env or the environment.")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


### step3 - 벡터DB에 필요한 열만 남기기

In [6]:
# ==== ChromaDB 적재용 데이터 준비 ====

# 1. 사용할 열 선택
chroma_columns = [
    # 식별자 및 임베딩 본문
    "museum_id",
    "text",

    # 시설 기본 정보
    "name",
    "ownership_type",
    "sido",
    "sigungu",
    "address",
    "latitude",
    "longitude",
    "homepage_url",

    # 관람료 및 조건 검색
    "adult_fee",
    "youth_fee",
    "child_fee",
    "adult_fee_status",
    "fee_notes",

    # 운영 정보
    "weekday_open",
    "weekday_close",
    "holiday_open",
    "holiday_close",
    "closure_rule_raw",

    # 편의시설 및 안내
    "amenities_raw",
    "transport_notes",
    "image_url",
    "image_copyright",

    # 출처 및 품질 관리
    "source_types",
    "requires_manual_review",
]

# 2. 필요한 열만 추출 (원본 df 유지)
df_chroma = df[chroma_columns].copy()

# 3. 준비된 데이터 저장
df_chroma.to_csv(
    "museum_chroma_ready.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. 상위 2개 행 확인
display(df_chroma.head(2))

,museum_id,text,name,ownership_type,sido,sigungu,address,latitude,longitude,homepage_url,...,weekday_close,holiday_open,holiday_close,closure_rule_raw,amenities_raw,transport_notes,image_url,image_copyright,source_types,requires_manual_review
0,M0001,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,박진전쟁기념관,공립,경상남도,창녕군,경상남도 창녕군 남지읍 월상길 27,35.447968,128.386193,https://www.cng.go.kr/01656/03204/03205.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/99/...,Type1,TOUR_API,True
1,M0003,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,창녕박물관,공립,경상남도,창녕군,경상남도 창녕군 창녕읍 창밀로 34,35.545406,128.505441,https://www.cng.go.kr/01656/01671.web,...,18:00,09:00,18:00,매주 월요일+1월 1일+설날+추석날,장애인화장실+유모차/휠체어 대여+유모차/휠체어 이용가능,NaN,https://tong.visitkorea.or.kr/cms/resource/15/...,Type1,TOUR_API,True


### step4 - text 열 임베딩 처리하기 + 적재

- museum_id만 메타데이터가 아니라 ChromaDB의 고유 ID로 저장 (ids로 저장됨)

In [7]:
# ==== ChromaDB 임베딩 및 영구 저장 ====

from pathlib import Path
import pandas as pd
import numpy as np
import chromadb

# 1. 프로젝트 루트 자동 탐색
NOTEBOOK_REL = Path(
    "src/for_build/build_vertorDB/vectoDB_builder.ipynb"
)

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / NOTEBOOK_REL).is_file()),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"프로젝트 루트를 찾을 수 없습니다. 현재 경로: {cwd}"
    )

# 2. ChromaDB 저장 위치
DB_PATH = PROJECT_ROOT / "data" / "vector_db" / "museum_chroma"
COLLECTION_NAME = "museum_documents"

DB_PATH.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(DB_PATH))
collection = client.get_or_create_collection(
    name=COLLECTION_NAME
)

# 3. 데이터 준비
df_ready = df_chroma.copy()

assert df_ready["museum_id"].notna().all()
assert df_ready["museum_id"].is_unique
assert df_ready["text"].notna().all()
assert df_ready["text"].apply(
    lambda x: isinstance(x, str) and bool(x.strip())
).all()

ids = df_ready["museum_id"].astype(str).tolist()
documents = df_ready["text"].tolist()

# 4. 메타데이터 변환
NUMERIC_COLS = {
    "latitude", "longitude",
    "adult_fee", "youth_fee", "child_fee"
}

def make_metadata(row):
    metadata = {}

    for col, value in row.items():
        if col in ("museum_id", "text"):
            continue

        if pd.isna(value):
            continue

        if isinstance(value, str):
            value = value.strip()
            if not value:
                continue

        if col in NUMERIC_COLS:
            value = pd.to_numeric(value, errors="coerce")
            if pd.isna(value) or not np.isfinite(value):
                continue
            value = float(value)
            if col not in ("latitude", "longitude") and value.is_integer():
                value = int(value)

        elif col == "requires_manual_review":
            if isinstance(value, str):
                if value.lower() not in ("true", "false"):
                    continue
                value = value.lower() == "true"
            else:
                value = bool(value)

        elif isinstance(value, (bool, np.bool_)):
            value = bool(value)

        elif isinstance(value, (int, np.integer)):
            value = int(value)

        elif isinstance(value, (float, np.floating)):
            value = float(value)

        else:
            value = str(value)

        metadata[col] = value

    return metadata

metadatas = [
    make_metadata(row)
    for _, row in df_ready.iterrows()
]

# 5. 기존 적재 데이터 확인
existing = collection.get(
    ids=ids,
    include=["documents", "metadatas"]
)

existing_map = {
    doc_id: (doc, meta or {})
    for doc_id, doc, meta in zip(
        existing["ids"],
        existing["documents"],
        existing["metadatas"]
    )
}

# 새 문서 또는 본문 변경
embed_idx = [
    i for i, doc_id in enumerate(ids)
    if doc_id not in existing_map
    or existing_map[doc_id][0] != documents[i]
]

# 본문은 같고 메타데이터만 변경
metadata_idx = [
    i for i, doc_id in enumerate(ids)
    if doc_id in existing_map
    and existing_map[doc_id][0] == documents[i]
    and existing_map[doc_id][1] != metadatas[i]
]

# 6. 필요한 문서만 OpenAI 임베딩
if embed_idx:
    vectors = embeddings.embed_documents(
        [documents[i] for i in embed_idx]
    )

    collection.upsert(
        ids=[ids[i] for i in embed_idx],
        embeddings=vectors,
        documents=[documents[i] for i in embed_idx],
        metadatas=[metadatas[i] for i in embed_idx]
    )

# 7. 메타데이터만 변경된 문서
if metadata_idx:
    collection.update(
        ids=[ids[i] for i in metadata_idx],
        metadatas=[metadatas[i] for i in metadata_idx]
    )

# 8. 저장 결과 확인
print(f"DB 저장 위치: {DB_PATH}")
print(f"Collection: {COLLECTION_NAME}")
print(f"임베딩 생성/갱신: {len(embed_idx)}건")
print(f"메타데이터 갱신: {len(metadata_idx)}건")
print(f"전체 적재 건수: {collection.count()}건")

DB 저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
Collection: museum_documents
임베딩 생성/갱신: 738건
메타데이터 갱신: 0건
전체 적재 건수: 738건


### step5 - 저장된 DB 확인

In [8]:
# ==== 저장된 ChromaDB 재조회 및 적재 검증 ====

from IPython.display import display

# 1. 디스크에 저장된 DB에 다시 연결
check_client = chromadb.PersistentClient(
    path=str(DB_PATH)
)

check_collection = check_client.get_collection(
    name=COLLECTION_NAME
)

# 2. 적재 건수 및 ID 검증
stored_count = check_collection.count()

assert stored_count == len(df_ready), (
    f"건수 불일치: {stored_count} / {len(df_ready)}"
)

stored_ids = set(check_collection.get(include=[])["ids"])
assert stored_ids == set(ids), "저장된 museum_id가 원본과 다릅니다."

# 3. 첫 2개 문서와 임베딩 조회
result = check_collection.get(
    ids=ids[:2],
    include=["documents", "metadatas", "embeddings"]
)

assert len(result["ids"]) == 2
assert result["embeddings"] is not None

# 4. 확인용 테이블
check_df = pd.DataFrame([
    {
        "museum_id": doc_id,
        "name": meta.get("name"),
        "text": doc[:100],
        "embedding_dim": len(vector)
    }
    for doc_id, doc, meta, vector in zip(
        result["ids"],
        result["documents"],
        result["metadatas"],
        result["embeddings"]
    )
])

print(f"저장 위치: {DB_PATH}")
print(f"저장된 문서: {stored_count}건")
print(f"임베딩 차원: {len(result['embeddings'][0])}")
print("ChromaDB 재조회 및 적재 검증 성공")

display(check_df.head(2))

저장 위치: C:\Users\Playdata\Desktop\mle-01-p3-team1\data\vector_db\museum_chroma
저장된 문서: 738건
임베딩 차원: 1536
ChromaDB 재조회 및 적재 검증 성공


,museum_id,name,text,embedding_dim
0,M0001,박진전쟁기념관,박진전쟁기념관은 6·25전쟁 최후의 보루였던 박진전투 현장을 재현하여 민족 분단의 ...,1536
1,M0003,창녕박물관,창녕박물관은 1996년 창녕지역 고분군에서 출토된 유물들을 전시할 목적으로 창녕유물...,1536


### step 6 - 검색으로 테스트 해보기

In [10]:
# ==== ChromaDB 의미 검색 테스트 (Top 3) ====

# 1. 사용자 질문 입력
query = input("검색할 질문을 입력하세요: ").strip()

if not query:
    raise ValueError("검색 질문을 입력해야 합니다.")

# 2. 질문 임베딩
query_vector = embeddings.embed_query(query)

# 3. ChromaDB 유사도 검색
results = check_collection.query(
    query_embeddings=[query_vector],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# 4. 원래 질문 출력
print("\n" + "=" * 70)
print(f"🔎 원래 질문: {query}")
print("=" * 70)

# 5. 상위 3개 검색 결과 출력
for i, (museum_id, document, metadata, distance) in enumerate(
    zip(
        results["ids"][0],
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    metadata = metadata or {}

    print(f"\n{'=' * 70}")
    print(f"📌 검색 결과 {i}위")
    print(f"{'=' * 70}")

    print(f"박물관명 : {metadata.get('name', '정보 없음')}")
    print(f"박물관 ID: {museum_id}")
    print(f"지역     : {metadata.get('sido', '')} {metadata.get('sigungu', '')}")
    print(f"벡터 거리: {distance:.4f}")

    print("\n[검색된 원본 텍스트]")
    print(document)

print(f"\n✅ 총 {len(results['ids'][0])}개 검색 완료")


🔎 원래 질문: 일제강점기 독립운동과 관련된 박물관

📌 검색 결과 1위
박물관명 : 몽양기념관
박물관 ID: M0875
지역     : 경기도 양평군
벡터 거리: 0.9219

[검색된 원본 텍스트]
2011년 11월 27일에 개관. 독립 ·평화통일운동가 여운형 관련 역사박물관

📌 검색 결과 2위
박물관명 : 경상북도독립운동기념관
박물관 ID: M0191
지역     : 경상북도 안동시
벡터 거리: 1.0089

[검색된 원본 텍스트]
경상북도독립운동기념관은 2007년 개관한 안동독립운동기념관을 2014년에 경상북도 전체의 기념관으로 승격·확장하여 2017년 6월 30일 다시 문을 열었다. 경북독립운동사에 관한 자료를 수집‧보존, 조사‧연구, 전시, 교육함으로써 민족의 자주독립을 지켜온 독립운동가들의 나라 사랑 정신을 계승하고 민족문화의 정체성 확립 및 올바른 국가관을 정립하는데 목적을 두고 있다. 또한 경상북도독립운동기념관의 전시실은 독립관과 의열관으로 구성되어 있으며, 1894년 갑오의병부터 1945년 조국 광복 때까지 51년간 펼쳐진 경북 사람들의 국내외 독립운동이 담겨 있다. 의병항쟁을 시작으로 국채보상운동, 자정순국, 만주지역 항일투쟁, 6·10 만세운동, 의열투쟁, 한국광복군 등 경북 사람들의 쉼 없는 항일투쟁과 관련 유물이 소개되어 있다. 또한 의열관은 안동 독립운동의 뿌리가 된 전통마을의 항일투쟁을 한눈에 볼 수 있다. 안동 독립운동의 뿌리가 된 전통마을의 항일투쟁을 전시하고 있는 전시실과 되살아나는 불씨, 나라 위해 목숨 바친 경북 사람들의 51년 독립운동을 영상으로 보여주는 추강영상실, 유아를 위한 교육과 체험공간인 새싹교육실 등으로 구성되어 있다. 경북 지역 독립운동의 역사를 기록하고 기념하는 시설이다. 의병 활동에서 광복에 이르는 지역의 항일운동과 독립운동가들의 활동을 전시·교육을 통해 소개한다.

📌 검색 결과 3위
박물관명 : 국립낙동강생물자원관
박물관 ID: M0519
지역     : 경상북도 상주시
벡터 거리: 1.0385



In [11]:
# ==== 벡터 검색 Top 10 및 원본 데이터 일치 검증 ====

import pandas as pd
from IPython.display import display

# 기존 질문 벡터로 상위 10개 조회
result = check_collection.query(
    query_embeddings=[query_vector],
    n_results=10,
    include=["documents", "metadatas", "distances"]
)

# 원본 데이터 매핑
original = df_chroma.set_index("museum_id")

rows = []

for i, (mid, doc, meta, dist) in enumerate(zip(
    result["ids"][0],
    result["documents"][0],
    result["metadatas"][0],
    result["distances"][0]
), start=1):

    # 원본 CSV와 저장된 본문 비교
    is_matched = (
        mid in original.index
        and original.loc[mid, "text"] == doc
        and original.loc[mid, "name"] == meta.get("name")
    )

    rows.append({
        "순위": i,
        "museum_id": mid,
        "박물관명": meta.get("name"),
        "거리": round(float(dist), 4),
        "원본 일치": is_matched,
        "본문 미리보기": doc[:100].replace("\n", " ")
    })

print(f"검색 질문: {query}")
display(pd.DataFrame(rows))

검색 질문: 일제강점기 독립운동과 관련된 박물관


,순위,museum_id,박물관명,거리,원본 일치,본문 미리보기
0,1,M0875,몽양기념관,0.9219,True,2011년 11월 27일에 개관. 독립 ·평화통일운동가 여운형 관련 역사박물관
1,2,M0191,경상북도독립운동기념관,1.0089,True,경상북도독립운동기념관은 2007년 개관한 안동독립운동기념관을 2014년에 경상북도 ...
2,3,M0519,국립낙동강생물자원관,1.0385,True,국립낙동강생물자원관은 담수(淡水) 분야에서 국가 생물주권을 확보하고 생물 다양성의 ...
3,4,M0525,국립관세박물관,1.0758,True,"국립관세박물관은 강남구에 위치한 국립박물관으로, 개항기부터 현대까지 한국 경제사와 ..."
4,5,M0840,경기도박물관,1.0765,True,경기도어린이박물관과 이웃하고 있는 경기도박물관은 경기도의 역사와 전통문화를 계승하고...
5,6,M0502,국립일제강제동원역사관,1.0772,True,국립일제강제동원역사관은 일제 강점기 강제동원의 실상을 규명함으로써 성숙한 역사의식을...
6,7,M0453,경북대학교 박물관,1.0780,True,"경북대학교 박물관은 1959년 개관 이래 유물의 수집과 보존, 전시, 연구 등을 통..."
7,8,M0930,강릉자수박물관,1.0821,True,강릉 오죽헌의 예술창작인촌에 위치한 강릉자수박물관은 자수에 새겨진 옛 여인들의 아름...
8,9,M0363,한림대학교박물관,1.0894,True,"한림대학교 박물관은 선사와 역사시대의 유물과 민속품 등을 수집 정리하고, 보전과 전..."
9,10,M0151,밀양시립박물관,1.0905,True,"밀양시립박물관은 낙동강유역의 문화를 집대성하여 효율적인 보존·전시함은 물론, 향토사..."


In [12]:
# ==== 주요 관련 박물관의 실제 검색 순위 확인 ====

import pandas as pd
from IPython.display import display

query = "일제강점기 독립운동과 관련된 박물관"

# 현재 질문으로 임베딩을 새로 생성해 이전 질문과 혼동 방지
query_vector = embeddings.embed_query(query)

# 전체 후보의 유사도 순위 조회
result = check_collection.query(
    query_embeddings=[query_vector],
    n_results=check_collection.count(),
    include=["metadatas", "distances"]
)

# 전체 순위 구성
ranking = {
    mid: {
        "rank": i,
        "distance": float(dist)
    }
    for i, (mid, dist) in enumerate(
        zip(result["ids"][0], result["distances"][0]),
        start=1
    )
}

# 실제로 관련성이 높은 것으로 예상되는 박물관
target_ids = [
    "M0875",  # 몽양기념관
    "M0191",  # 경상북도독립운동기념관
    "M0714",  # 독립기념관
    "M0253",  # 안중근의사기념관
    "M0503",  # 대한민국임시정부기념관
    "M0502",  # 국립일제강제동원역사관
    "M0151",  # 밀양시립박물관
]

metadata_by_id = {
    mid: meta
    for mid, meta in zip(
        result["ids"][0],
        result["metadatas"][0]
    )
}

rows = []

for mid in target_ids:
    if mid not in ranking:
        continue

    rows.append({
        "검색 순위": ranking[mid]["rank"],
        "박물관": metadata_by_id[mid].get("name"),
        "museum_id": mid,
        "거리": round(ranking[mid]["distance"], 4)
    })

print(f"원래 질문: {query}")
display(
    pd.DataFrame(rows).sort_values("검색 순위")
)

원래 질문: 일제강점기 독립운동과 관련된 박물관


,검색 순위,박물관,museum_id,거리
0,1,몽양기념관,M0875,0.9220
1,2,경상북도독립운동기념관,M0191,1.0088
5,6,국립일제강제동원역사관,M0502,1.0770
6,10,밀양시립박물관,M0151,1.0903
2,21,독립기념관,M0714,1.1140
4,410,국립대한민국임시정부기념관,M0503,1.3943
3,562,안중근의사기념관,M0253,1.4730


## step 0 - 검색결과의 불만족으로 인한 테스트 진행

In [13]:

# ==== 임베딩 방식 3종 비교 테스트 (DB 재적재 없음) ====

import numpy as np
import pandas as pd
from IPython.display import display

# 1. 실험 설정
QUERY = "일제강점기 독립운동과 관련된 박물관"
TITLE_WEIGHT = 0.35
TOP_K = 10

# 2. 기존 ChromaDB에서 데이터 및 임베딩 조회
data = check_collection.get(
    include=["documents", "metadatas", "embeddings"]
)

ids = data["ids"]
names = [
    meta.get("name", "")
    for meta in data["metadatas"]
]
texts = data["documents"]

# 기존 본문 임베딩 재사용
body_vectors = np.asarray(
    data["embeddings"], dtype=np.float32
)

assert len(ids) == check_collection.count()
assert len(set(ids)) == len(ids)

# 3. 새로 필요한 임베딩만 생성
# B: 제목 + 본문
combined_texts = [
    f"박물관명: {name}\n박물관 설명: {text}"
    for name, text in zip(names, texts)
]

combined_vectors = np.asarray(
    embeddings.embed_documents(combined_texts),
    dtype=np.float32
)

# C: 제목만 별도 임베딩
title_texts = [
    f"박물관명: {name}"
    for name in names
]

title_vectors = np.asarray(
    embeddings.embed_documents(title_texts),
    dtype=np.float32
)

# 4. 벡터 정규화 함수
def normalize(vectors):
    vectors = np.asarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(
        vectors, axis=-1, keepdims=True
    )
    return vectors / np.maximum(norms, 1e-12)

# C: 제목 35% + 본문 65%
weighted_vectors = normalize(
    TITLE_WEIGHT * normalize(title_vectors)
    + (1 - TITLE_WEIGHT) * normalize(body_vectors)
)

# 5. 세 방식의 벡터 준비
experiments = {
    "A. 본문 단독": body_vectors,
    "B. 제목 + 본문": combined_vectors,
    "C. 제목 35% + 본문 65%": weighted_vectors
}

# 6. 고정 질문 임베딩 (한 번만 호출)
query_vector = normalize(
    np.asarray(embeddings.embed_query(QUERY))
)

# 7. 코사인 유사도 기반 전체 순위 계산
rankings = {}
scores_by_method = {}

for method, vectors in experiments.items():
    normalized_vectors = normalize(vectors)

    if normalized_vectors.shape[1] != query_vector.shape[0]:
        raise ValueError(f"{method}: 임베딩 차원 불일치")

    scores = normalized_vectors @ query_vector
    ordered_indices = np.argsort(-scores)

    rankings[method] = {
        ids[idx]: rank
        for rank, idx in enumerate(
            ordered_indices, start=1
        )
    }
    scores_by_method[method] = scores

    # 상위 10개 출력
    top_rows = [
        {
            "순위": rank,
            "museum_id": ids[idx],
            "박물관": names[idx],
            "코사인 유사도": round(float(scores[idx]), 4)
        }
        for rank, idx in enumerate(
            ordered_indices[:TOP_K], start=1
        )
    ]

    print(f"\n{'=' * 65}")
    print(f"{method} - TOP {TOP_K}")
    print(f"{'=' * 65}")
    display(pd.DataFrame(top_rows))

# 8. 주요 관련 박물관 순위 비교
target_ids = {
    "M0875": "몽양기념관",
    "M0191": "경상북도독립운동기념관",
    "M0714": "독립기념관",
    "M0503": "국립대한민국임시정부기념관",
    "M0253": "안중근의사기념관",
    "M0502": "국립일제강제동원역사관",
    "M0151": "밀양시립박물관"
}

comparison = []

for mid, name in target_ids.items():
    if mid not in rankings["A. 본문 단독"]:
        continue

    rank_a = rankings["A. 본문 단독"][mid]
    rank_b = rankings["B. 제목 + 본문"][mid]
    rank_c = rankings["C. 제목 35% + 본문 65%"][mid]

    comparison.append({
        "박물관": name,
        "기존 A": rank_a,
        "결합 B": rank_b,
        "가중 C": rank_c,
        "B 순위 상승": rank_a - rank_b,
        "C 순위 상승": rank_a - rank_c
    })

print("\n" + "=" * 65)
print(f"원래 질문: {QUERY}")
print("주요 박물관 검색 순위 비교")
print("=" * 65)

display(
    pd.DataFrame(comparison).sort_values("기존 A")
)

print("\n✅ 세 방식 비교 완료 (ChromaDB 변경 없음)")



A. 본문 단독 - TOP 10


,순위,museum_id,박물관,코사인 유사도
0,1,M0875,몽양기념관,0.5392
1,2,M0191,경상북도독립운동기념관,0.4958
2,3,M0519,국립낙동강생물자원관,0.4809
3,4,M0525,국립관세박물관,0.4623
4,5,M0840,경기도박물관,0.4619
5,6,M0502,국립일제강제동원역사관,0.4617
6,7,M0453,경북대학교 박물관,0.4611
7,8,M0930,강릉자수박물관,0.4588
8,9,M0363,한림대학교박물관,0.4556
9,10,M0151,밀양시립박물관,0.4551



B. 제목 + 본문 - TOP 10


,순위,museum_id,박물관,코사인 유사도
0,1,M0191,경상북도독립운동기념관,0.5581
1,2,M0519,국립낙동강생물자원관,0.5468
2,3,M0875,몽양기념관,0.5393
3,4,M0502,국립일제강제동원역사관,0.5307
4,5,M0738,운강이강년기념관,0.5097
5,6,M0840,경기도박물관,0.4841
6,7,M0714,독립기념관,0.4829
7,8,M0525,국립관세박물관,0.4805
8,9,M1015,양구선사박물관,0.4752
9,10,M0358,강아지숲 박물관,0.4736



C. 제목 35% + 본문 65% - TOP 10


,순위,museum_id,박물관,코사인 유사도
0,1,M0191,경상북도독립운동기념관,0.5792
1,2,M0875,몽양기념관,0.5694
2,3,M0519,국립낙동강생물자원관,0.5617
3,4,M0502,국립일제강제동원역사관,0.5607
4,5,M0714,독립기념관,0.5424
5,6,M0738,운강이강년기념관,0.5368
6,7,M1012,건국대학교박물관,0.5252
7,8,M0840,경기도박물관,0.5226
8,9,M0484,국립부여박물관,0.5196
9,10,M0525,국립관세박물관,0.5152



원래 질문: 일제강점기 독립운동과 관련된 박물관
주요 박물관 검색 순위 비교


,박물관,기존 A,결합 B,가중 C,B 순위 상승,C 순위 상승
0,몽양기념관,1,3,2,-2,-1
1,경상북도독립운동기념관,2,1,1,1,1
5,국립일제강제동원역사관,6,4,4,2,2
6,밀양시립박물관,10,29,40,-19,-30
2,독립기념관,21,7,5,14,16
3,국립대한민국임시정부기념관,409,68,293,341,116
4,안중근의사기념관,562,555,526,7,36



✅ 세 방식 비교 완료 (ChromaDB 변경 없음)


## step0 - 검색어 확장 테스트

In [14]:

# ==== Ground Truth 기반 검색어 확장 성능 평가 ====

import numpy as np
import pandas as pd
from IPython.display import display

# --------------------------------------------------
# 1. 정답 데이터: 2=직접 관련, 1=부분 관련
# --------------------------------------------------
GOLD = {
    "M0875": 2,  # 몽양기념관
    "M0191": 2,  # 경상북도독립운동기념관
    "M0714": 2,  # 독립기념관
    "M0503": 2,  # 국립대한민국임시정부기념관
    "M0253": 2,  # 안중근의사기념관
    "M0556": 2,  # 화성시독립운동기념관
    "M0716": 2,  # 유관순열사기념관
    "M1001": 2,  # 안성3.1운동기념관

    "M0151": 1,  # 밀양시립박물관
    "M0502": 1,  # 국립일제강제동원역사관
    "M0738": 1,  # 운강이강년기념관
}

STRONG_GOLD = {mid for mid, grade in GOLD.items() if grade == 2}

ids = list(ids)
names = list(names)
id_to_name = dict(zip(ids, names))
n = len(ids)

assert set(GOLD).issubset(set(ids)), "DB에 없는 정답 ID가 있습니다."

# --------------------------------------------------
# 2. 고정 질문과 검색어 확장
# --------------------------------------------------
QUERIES = {
    "원문":
        "일제강점기 독립운동과 관련된 박물관",

    "통합 확장":
        "일제강점기 항일 독립운동, 3·1운동, 대한민국 임시정부, "
        "광복운동과 독립운동가의 활동을 다룬 박물관 및 기념관",

    "항일운동":
        "일제 식민통치에 저항한 항일 독립운동과 만세운동을 "
        "전시하는 역사박물관",

    "임시정부":
        "3·1운동과 대한민국 임시정부, 광복운동의 역사와 "
        "관련 유물을 소개하는 기념관",

    "독립운동가":
        "국권 회복을 위한 독립운동가의 항일 의거, "
        "순국과 희생을 기념하는 박물관",
}

# Multi-query RRF에서 원래 질문 비중을 조금 높임
MULTI_WEIGHTS = {
    "원문": 0.4,
    "항일운동": 0.2,
    "임시정부": 0.2,
    "독립운동가": 0.2,
}

RRF_K = 60

# --------------------------------------------------
# 3. 정규화 및 질문 임베딩
# --------------------------------------------------
def unit(v):
    v = np.asarray(v, dtype=np.float32)
    norm = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.maximum(norm, 1e-12)

# 질문 5개만 새로 임베딩
query_vectors = {
    key: unit(embeddings.embed_query(text))
    for key, text in QUERIES.items()
}

# --------------------------------------------------
# 4. 평가 지표
# --------------------------------------------------
def get_rank_map(order):
    return {
        ids[idx]: rank
        for rank, idx in enumerate(order, start=1)
    }

def known_recall(order, k):
    """지정한 직접 관련 정답 중 Top-K에 포함된 비율"""
    found = {ids[i] for i in order[:k]}
    return len(found & STRONG_GOLD) / len(STRONG_GOLD)

def ndcg_at_k(order, k=10):
    """관련성 2/1/0을 사용한 임시 nDCG@K"""
    grades = np.array(
        [GOLD.get(ids[i], 0) for i in order[:k]],
        dtype=float
    )

    ideal = np.array(
        sorted(GOLD.values(), reverse=True)[:k],
        dtype=float
    )

    positions = np.arange(2, len(grades) + 2)
    ideal_positions = np.arange(2, len(ideal) + 2)

    dcg = np.sum((2**grades - 1) / np.log2(positions))
    idcg = np.sum((2**ideal - 1) / np.log2(ideal_positions))

    return float(dcg / idcg) if idcg > 0 else 0.0

# --------------------------------------------------
# 5. A/B/C × 검색 전략 3종 비교
# --------------------------------------------------
all_orders = {}
summary_rows = []

for model_name, doc_vectors in experiments.items():

    doc_vectors = unit(doc_vectors)

    assert doc_vectors.shape[0] == n, "ID와 벡터 개수 불일치"

    # 각 확장 질문에 대한 코사인 유사도
    scores = {
        key: doc_vectors @ qvec
        for key, qvec in query_vectors.items()
    }

    # 전략 1: 원래 질문
    original_order = np.argsort(
        -scores["원문"], kind="stable"
    )

    # 전략 2: 통합 확장 질문 하나
    expanded_order = np.argsort(
        -scores["통합 확장"], kind="stable"
    )

    # 전략 3: 여러 질문의 순위를 RRF로 통합
    rrf_scores = np.zeros(n, dtype=float)

    for key, weight in MULTI_WEIGHTS.items():
        order = np.argsort(-scores[key], kind="stable")

        ranks = np.empty(n, dtype=int)
        ranks[order] = np.arange(1, n + 1)

        rrf_scores += weight / (RRF_K + ranks)

    multi_order = np.argsort(
        -rrf_scores, kind="stable"
    )

    strategies = {
        "원문 검색": original_order,
        "단일 확장": expanded_order,
        "Multi-query RRF": multi_order,
    }

    for strategy, order in strategies.items():
        all_orders[(model_name, strategy)] = order

        top10_ids = {ids[i] for i in order[:10]}

        summary_rows.append({
            "임베딩": model_name,
            "검색 전략": strategy,
            "Recall@5(%)": round(
                known_recall(order, 5) * 100, 1
            ),
            "Recall@10(%)": round(
                known_recall(order, 10) * 100, 1
            ),
            "Recall@30(%)": round(
                known_recall(order, 30) * 100, 1
            ),
            "nDCG@10": round(ndcg_at_k(order), 4),
            "낙동강생물자원관 Top10": "포함" if "M0519" in top10_ids else "제외",
        })

# --------------------------------------------------
# 6. 전체 성능 비교표
# --------------------------------------------------
summary_df = pd.DataFrame(summary_rows)

print("=" * 75)
print("검색 질문:", QUERIES["원문"])
print("직접 관련 정답:", len(STRONG_GOLD), "개")
print("부분 관련 정답:", len(GOLD) - len(STRONG_GOLD), "개")
print("=" * 75)

display(summary_df)

# --------------------------------------------------
# 7. 기존 대비 검색어 확장 개선량
# --------------------------------------------------
improvement_rows = []

for model_name in experiments:
    base = summary_df[
        (summary_df["임베딩"] == model_name)
        & (summary_df["검색 전략"] == "원문 검색")
    ].iloc[0]

    for strategy in ["단일 확장", "Multi-query RRF"]:
        current = summary_df[
            (summary_df["임베딩"] == model_name)
            & (summary_df["검색 전략"] == strategy)
        ].iloc[0]

        improvement_rows.append({
            "임베딩": model_name,
            "확장 전략": strategy,
            "Recall@10 변화(pp)": round(
                current["Recall@10(%)"] - base["Recall@10(%)"], 1
            ),
            "Recall@30 변화(pp)": round(
                current["Recall@30(%)"] - base["Recall@30(%)"], 1
            ),
            "nDCG@10 변화": round(
                current["nDCG@10"] - base["nDCG@10"], 4
            ),
        })

print("\n[원문 검색 대비 개선량]")
display(pd.DataFrame(improvement_rows))

# --------------------------------------------------
# 8. B 방식의 정답 박물관별 순위 비교
# --------------------------------------------------
B_NAME = "B. 제목 + 본문"

if B_NAME in experiments:
    strategies = [
        "원문 검색",
        "단일 확장",
        "Multi-query RRF",
    ]

    rank_maps = {
        strategy: get_rank_map(all_orders[(B_NAME, strategy)])
        for strategy in strategies
    }

    rows = []

    for mid, grade in GOLD.items():
        old_rank = rank_maps["원문 검색"][mid]
        single_rank = rank_maps["단일 확장"][mid]
        multi_rank = rank_maps["Multi-query RRF"][mid]

        rows.append({
            "박물관": id_to_name[mid],
            "관련성": "직접" if grade == 2 else "부분",
            "기존 순위": old_rank,
            "단일 확장": single_rank,
            "Multi-query": multi_rank,
            "단일 순위 상승": old_rank - single_rank,
            "Multi 순위 상승": old_rank - multi_rank,
        })

    print("\n[B 방식: 박물관별 순위 변화]")
    display(
        pd.DataFrame(rows).sort_values("기존 순위")
    )

    # Multi-query Top 10도 출력
    order = all_orders[(B_NAME, "Multi-query RRF")]

    print("\n[B 방식: Multi-query 최종 Top 10]")
    display(pd.DataFrame([
        {
            "순위": rank,
            "museum_id": ids[idx],
            "박물관": names[idx],
            "정답 등급": {
                2: "직접",
                1: "부분",
                0: "미평가"
            }[GOLD.get(ids[idx], 0)]
        }
        for rank, idx in enumerate(order[:10], start=1)
    ]))

print("\n평가 완료: 기존 ChromaDB와 문서 임베딩 변경 없음")


검색 질문: 일제강점기 독립운동과 관련된 박물관
직접 관련 정답: 8 개
부분 관련 정답: 3 개


,임베딩,검색 전략,Recall@5(%),Recall@10(%),Recall@30(%),nDCG@10,낙동강생물자원관 Top10
0,A. 본문 단독,원문 검색,25.0,25.0,37.5,0.4448,포함
1,A. 본문 단독,단일 확장,37.5,62.5,75.0,0.6931,제외
2,A. 본문 단독,Multi-query RRF,37.5,37.5,37.5,0.5748,포함
3,B. 제목 + 본문,원문 검색,25.0,37.5,50.0,0.5074,포함
4,B. 제목 + 본문,단일 확장,37.5,75.0,75.0,0.7890,포함
5,B. 제목 + 본문,Multi-query RRF,37.5,50.0,75.0,0.6594,포함
6,C. 제목 35% + 본문 65%,원문 검색,37.5,37.5,50.0,0.5494,포함
7,C. 제목 35% + 본문 65%,단일 확장,50.0,62.5,75.0,0.7511,포함
8,C. 제목 35% + 본문 65%,Multi-query RRF,37.5,50.0,62.5,0.6424,포함



[원문 검색 대비 개선량]


,임베딩,확장 전략,Recall@10 변화(pp),Recall@30 변화(pp),nDCG@10 변화
0,A. 본문 단독,단일 확장,37.5,37.5,0.2483
1,A. 본문 단독,Multi-query RRF,12.5,0.0,0.1300
2,B. 제목 + 본문,단일 확장,37.5,25.0,0.2816
3,B. 제목 + 본문,Multi-query RRF,12.5,25.0,0.1520
4,C. 제목 35% + 본문 65%,단일 확장,25.0,25.0,0.2017
5,C. 제목 35% + 본문 65%,Multi-query RRF,12.5,12.5,0.0930



[B 방식: 박물관별 순위 변화]


,박물관,관련성,기존 순위,단일 확장,Multi-query,단일 순위 상승,Multi 순위 상승
1,경상북도독립운동기념관,직접,1,1,1,0,0
0,몽양기념관,직접,3,6,2,-3,1
9,국립일제강제동원역사관,부분,4,7,4,-3,0
10,운강이강년기념관,부분,5,5,5,0,0
2,독립기념관,직접,7,8,3,-1,4
5,화성시독립운동기념관,직접,18,9,7,9,11
8,밀양시립박물관,부분,29,28,16,1,13
3,국립대한민국임시정부기념관,직접,68,2,13,66,55
7,안성3.1운동기념관,직접,69,4,19,65,50
6,유관순열사기념관,직접,317,102,78,215,239



[B 방식: Multi-query 최종 Top 10]


,순위,museum_id,박물관,정답 등급
0,1,M0191,경상북도독립운동기념관,직접
1,2,M0875,몽양기념관,직접
2,3,M0714,독립기념관,직접
3,4,M0502,국립일제강제동원역사관,부분
4,5,M0738,운강이강년기념관,부분
5,6,M0519,국립낙동강생물자원관,미평가
6,7,M0556,화성시독립운동기념관,직접
7,8,M0490,국립농업박물관,미평가
8,9,M0484,국립부여박물관,미평가
9,10,M0473,자유수호평화박물관,미평가



평가 완료: 기존 ChromaDB와 문서 임베딩 변경 없음
